# 🤖 Sistema PQRSF Inteligente — Nivel 2: LangChain + Gemini

**Módulo:** Sistemas Inteligentes  
**Docente:** Luis Fernando Castillo  

---

Este notebook reimplementa en Python el mismo flujo del Nivel 1 (n8n), usando:

- **LangChain ≥ 0.2** como framework de agentes
- **Google Gemini** (gemini-1.5-flash) como modelo LLM
- **Google Sheets API** para persistencia de datos
- **Gmail API** (via gspread + google-auth) para notificaciones
- **Structured Output** para clasificación con esquema JSON definido

### Arquitectura del pipeline
```
Entrada del usuario
       ↓
1. Generar Radicado único
       ↓
2. Agente LangChain (Gemini)
   ├─ Tool: Enviar correo de confirmación (Gmail)
   └─ Structured Output: categoria, prioridad, area, resumen
       ↓
3. Verificar prioridad → si es Alta: notificar área Jurídica
       ↓
4. Registrar en Google Sheets
       ↓
Resultado final (radicado, categoría, área, prioridad)
```

## 📦 Celda 1 — Instalación de dependencias
Instalamos todas las librerías necesarias. Solo se ejecuta una vez.

In [17]:
# Instalamos LangChain, el conector de Gemini, Google Sheets y Gmail
!pip install -q \
    langchain \
    langchain-google-genai \
    langchain-core \
    langgraph \
    gspread \
    google-auth \
    google-auth-oauthlib \
    google-api-python-client \
    pydantic

print("✅ Dependencias instaladas correctamente")

✅ Dependencias instaladas correctamente


## 🔑 Celda 2 — Configuración de credenciales
Leemos la API Key de Gemini desde los **Secrets de Colab** (ícono 🔑 en el panel izquierdo).
El archivo JSON de Google Sheets se sube manualmente.

In [18]:
import os
import json
from google.colab import userdata, files

# --- API Key de Gemini ---
# Ve al panel izquierdo de Colab → ícono 🔑 "Secrets"
# Agrega un secret con nombre: GEMINI_API_KEY
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY
print("✅ Gemini API Key cargada")

# --- Credenciales de Google Sheets (JSON de cuenta de servicio) ---
# Sube el archivo JSON de tu cuenta de servicio cuando aparezca el botón
print("\n📂 Sube tu archivo JSON de credenciales de Google (cuenta de servicio):")
uploaded = files.upload()

# Guardamos la ruta del archivo subido
SERVICE_ACCOUNT_FILE = list(uploaded.keys())[0]
print(f"✅ Credenciales cargadas: {SERVICE_ACCOUNT_FILE}")

# --- Configuración del spreadsheet ---
# Cambia este valor por el ID de tu Google Sheet
# Lo encuentras en la URL: docs.google.com/spreadsheets/d/<<ESTE_ID>>/edit
SPREADSHEET_ID = "1OE-OUXsbz2ZURVzamnlIlJ-Ri5FOP2du3FzW54iVu5E"
SHEET_NAME = "Hoja 1"  # Nombre de la hoja dentro del spreadsheet

# --- Correo del área jurídica para alertas urgentes ---
CORREO_JURIDICA = "acalebe21@gmail.com"  # Cambia este correo
CORREO_REMITENTE = "acalebe21@gmail.com"   # Tu correo de Gmail configurado

✅ Gemini API Key cargada

📂 Sube tu archivo JSON de credenciales de Google (cuenta de servicio):


Saving service_account.json to service_account (1).json
✅ Credenciales cargadas: service_account (1).json


## 📚 Celda 3 — Importaciones y configuración del LLM
Configuramos el modelo Gemini con LangChain.

In [19]:
import random
import datetime
import gspread
from google.oauth2.service_account import Credentials
from googleapiclient.discovery import build
from google.oauth2 import service_account
import base64
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# LangChain — componentes principales
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser

# --- Configuración del LLM (Gemini 2.5 Flash) ---
# temperature=0 → respuestas deterministas y consistentes (ideal para clasificación)
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=GEMINI_API_KEY
)

print("✅ LLM Gemini 2.5 Flash configurado")
print(f"   Modelo: gemini-2.5-flash | Temperature: 0")

✅ LLM Gemini 2.5 Flash configurado
   Modelo: gemini-2.5-flash | Temperature: 0


## 🔢 Celda 4 — Generación de Radicado
Misma lógica que el nodo JavaScript de n8n, replicada en Python.
Formato: `PQRS-YYYYMMDD-timestamp-random`

In [20]:
def generar_radicado() -> dict:
    """
    Genera un número de radicado único para cada solicitud PQRS.
    Replica exactamente el nodo 'Generar Radicado' de n8n.

    Returns:
        dict con 'radicado' (str) y 'fecha' (str en formato YYYY-MM-DD)
    """
    now = datetime.datetime.now()

    # Formato de fecha: YYYYMMDD
    fecha_str = now.strftime("%Y%m%d")

    # Timestamp en milisegundos (equivalente a Date.now() en JavaScript)
    timestamp = int(now.timestamp() * 1000)

    # Número aleatorio de 3 dígitos (000-999)
    random_num = str(random.randint(0, 999)).zfill(3)

    # Construimos el radicado con el mismo formato que n8n
    radicado = f"PQRS-{fecha_str}-{timestamp}-{random_num}"

    # Fecha legible para registrar en la hoja
    fecha = now.strftime("%Y-%m-%d")

    return {"radicado": radicado, "fecha": fecha}

# Probamos la función
prueba = generar_radicado()
print("🧾 Ejemplo de radicado generado:")
print(f"   Radicado: {prueba['radicado']}")
print(f"   Fecha:    {prueba['fecha']}")

🧾 Ejemplo de radicado generado:
   Radicado: PQRS-20260608-1780933572143-844
   Fecha:    2026-06-08


## 📧 Celda 5 — Configuración de Gmail API
Usamos la Gmail API con las mismas credenciales JSON para enviar correos HTML.
Necesitamos el scope de Gmail además del de Sheets.

In [21]:
# Scopes necesarios: Sheets para leer/escribir, Gmail para enviar correos
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/gmail.send",
    "https://www.googleapis.com/auth/gmail.compose"
]

# Cargamos las credenciales del archivo JSON subido
creds = Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE,
    scopes=SCOPES
)

# Cliente de Google Sheets
gc = gspread.authorize(creds)


gmail_service = None
print("📨 Gmail en modo demo — los correos se mostrarán en consola")

print("✅ Google Sheets API configurada")


def enviar_correo(destinatario: str, asunto: str, cuerpo_html: str) -> bool:
    """
    Envía un correo HTML usando Gmail API.
    Si no hay servicio de Gmail disponible, simula el envío (modo demo).

    Returns:
        True si se envió correctamente, False en caso contrario
    """
    if gmail_service is None:
        # Modo demo: mostramos el correo en consola
        print(f"\n📨 [MODO DEMO] Correo que se enviaría a: {destinatario}")
        print(f"   Asunto: {asunto}")
        print(f"   Cuerpo (HTML): {cuerpo_html[:200]}...")
        return True

    try:
        # Construimos el mensaje MIME
        mensaje = MIMEMultipart('alternative')
        mensaje['to'] = destinatario
        mensaje['subject'] = asunto
        mensaje.attach(MIMEText(cuerpo_html, 'html'))

        # Codificamos en base64 para la API
        raw = base64.urlsafe_b64encode(mensaje.as_bytes()).decode()
        gmail_service.users().messages().send(
            userId='me',
            body={'raw': raw}
        ).execute()
        return True
    except Exception as e:
        print(f"❌ Error enviando correo: {e}")
        return False

print("✅ Función enviar_correo() lista")

📨 Gmail en modo demo — los correos se mostrarán en consola
✅ Google Sheets API configurada
✅ Función enviar_correo() lista


## 🛠️ Celda 6 — Definición de Tools para el Agente
En LangChain, las **Tools** son funciones que el agente puede llamar durante su razonamiento.
Aquí definimos la herramienta de Gmail, equivalente al nodo `Gmail Tool` de n8n.

In [22]:
# Variable global para que la tool pueda acceder al contexto de la solicitud actual
_contexto_actual = {}

@tool
def enviar_correo_confirmacion(destinatario: str, nombre: str, radicado: str,
                                categoria: str, prioridad: str, area: str) -> str:
    """
    Envía un correo HTML de confirmación al usuario que envió la solicitud PQRS.
    DEBES llamar esta herramienta siempre que proceses una solicitud PQRS.

    Args:
        destinatario: Correo electrónico del solicitante
        nombre: Nombre completo del solicitante
        radicado: Número de radicado único generado (ej: PQRS-20240615-...)
        categoria: Tipo de solicitud clasificada (Petición/Queja/Reclamo/Sugerencia)
        prioridad: Nivel de prioridad (Alta/Media/Baja)
        area: Área responsable asignada

    Returns:
        Mensaje confirmando si el correo fue enviado exitosamente
    """
    # Determinamos el tiempo de respuesta según la prioridad
    tiempos = {
        "Alta": "1-2 días hábiles",
        "Media": "3-5 días hábiles",
        "Baja": "5-10 días hábiles"
    }
    tiempo_respuesta = tiempos.get(prioridad, "3-5 días hábiles")

    # Colores según prioridad (igual que el frontend)
    colores = {"Alta": "#e74c3c", "Media": "#f39c12", "Baja": "#27ae60"}
    color_prioridad = colores.get(prioridad, "#3498db")

    # Construimos el HTML del correo (mismo diseño que n8n)
    cuerpo_html = f"""
    <html><body style="font-family: Arial, sans-serif; max-width: 600px; margin: 0 auto;">
        <div style="background: linear-gradient(135deg, #667eea, #764ba2); padding: 30px; border-radius: 10px 10px 0 0;">
            <h1 style="color: white; margin: 0;">✅ Solicitud Recibida</h1>
            <p style="color: rgba(255,255,255,0.9); margin: 5px 0 0;">Sistema PQRSF Inteligente</p>
        </div>
        <div style="background: #f8f9fa; padding: 30px; border-radius: 0 0 10px 10px;">
            <p>Estimado/a <strong>{nombre}</strong>,</p>
            <p>Hemos recibido su solicitud y ha sido procesada exitosamente.</p>

            <div style="background: white; padding: 20px; border-radius: 8px; border-left: 4px solid #667eea; margin: 20px 0;">
                <h3 style="margin-top: 0; color: #333;">📋 Detalles de su solicitud</h3>
                <table style="width: 100%; border-collapse: collapse;">
                    <tr><td style="padding: 8px; font-weight: bold; color: #666;">Radicado:</td>
                        <td style="padding: 8px; font-family: monospace; background: #f0f0f0; border-radius: 4px;">{radicado}</td></tr>
                    <tr><td style="padding: 8px; font-weight: bold; color: #666;">Categoría:</td>
                        <td style="padding: 8px;">{categoria}</td></tr>
                    <tr><td style="padding: 8px; font-weight: bold; color: #666;">Área responsable:</td>
                        <td style="padding: 8px;">{area}</td></tr>
                    <tr><td style="padding: 8px; font-weight: bold; color: #666;">Prioridad:</td>
                        <td style="padding: 8px;"><span style="background: {color_prioridad}; color: white; padding: 3px 10px; border-radius: 12px;">{prioridad}</span></td></tr>
                    <tr><td style="padding: 8px; font-weight: bold; color: #666;">Tiempo estimado:</td>
                        <td style="padding: 8px;">{tiempo_respuesta}</td></tr>
                </table>
            </div>

            <p>Gracias por comunicarse con nosotros.</p>
            <p style="color: #999; font-size: 12px;">Este es un correo automático generado por el Sistema PQRSF.</p>
        </div>
    </body></html>
    """

    asunto = f"Confirmación de recepción PQRS - {radicado}"
    exito = enviar_correo(destinatario, asunto, cuerpo_html)

    if exito:
        return f"✅ Correo de confirmación enviado exitosamente a {destinatario} con radicado {radicado}"
    else:
        return f"❌ No se pudo enviar el correo a {destinatario}"


# Lista de tools disponibles para el agente
tools = [enviar_correo_confirmacion]

print("✅ Tools definidas:")
for t in tools:
    print(f"   - {t.name}: {t.description[:60]}...")

✅ Tools definidas:
   - enviar_correo_confirmacion: Envía un correo HTML de confirmación al usuario que envió la...


## 💬 Celda 7 — ChatPromptTemplate y Esquema de Salida
Definimos el **system message** y el **human template** del agente.
Usamos **Pydantic** para definir el esquema de salida estructurado (equivalente al Structured Output Parser de n8n).

In [23]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# --- Esquema de salida estructurada con Pydantic ---
# Define exactamente los campos que debe retornar el agente
# Mismo esquema que el Structured Output Parser de n8n
class ClasificacionPQRS(BaseModel):
    categoria: str = Field(
        description="Tipo de solicitud: Petición, Queja, Reclamo o Sugerencia"
    )
    prioridad: str = Field(
        description="Nivel de urgencia: Alta, Media o Baja"
    )
    area: str = Field(
        description="Área responsable: Servicio al Cliente, Servicios Financieros, Operaciones, Jurídica o Innovación"
    )
    resumen: str = Field(
        description="Resumen breve del caso en máximo 100 palabras"
    )
    palabras_clave: list = Field(
        default=[],
        description="Lista de las palabras clave *específicamente definidas* como críticas (fraude, estafa, robo, demanda, lavado de activos) si alguna fue detectada. Si no, la lista debe estar vacía."
    )


# --- System Message del Agente ---
# Replica exactamente el system message del nodo Clasificador PQRS de n8n
SYSTEM_MESSAGE = """Eres un asistente especializado en la gestión de PQRS (Peticiones, Quejas, Reclamos y Sugerencias). Tienes una estricta directriz de **no inventar** valores para los campos de clasificación.

... Como parte crucial de tu proceso, tienes acceso a la herramienta de Gmail y DEBES usarla para enviar el correo de confirmación al usuario.

Áreas responsables y sus competencias:
- Servicio al Cliente: Atención general, consultas, información
- Servicios Financieros: Productos financieros, cuentas, transacciones
- Operaciones: Procesos internos, logística, infraestructura
- Jurídica: Asuntos legales, fraude, demandas, cumplimiento, *siempre que se detecten palabras críticas*.
- Innovación: Sugerencias de mejora, nuevas tecnologías, optimización de procesos

Reglas de clasificación (estrictamente por orden de importancia):
1. SIEMPRE que se detecte ALGUNA de las siguientes palabras clave en la descripción o asunto:
   **fraude, estafa, robo, demanda, lavado de activos**,
   ENTONCES la prioridad DEBE SER 'Alta' y el área DEBE SER 'Jurídica'.
   NO HAY, BAJO NINGUNA CIRCUNSTANCIA, EXCEPCIONES A ESTA REGLA. Ignora cualquier otra interpretación.
   Además, el campo `palabras_clave` en el JSON debe contener *SOLAMENTE* las palabras de esta lista que se hayan detectado. No incluyas palabras que no estén en esta lista.
2. Las únicas áreas válidas son: 'Servicio al Cliente', 'Servicios Financieros', 'Operaciones', 'Jurídica' o 'Innovación'. **NO INVENTES NI USES OTRAS ÁREAS BAJO NINGÚN CONCEPTO.**
3. Palabras como 'urgencia', 'cuenta bancaria', 'demora' NO son críticas para la clasificación de prioridad. NO fuerzan la prioridad 'Alta' ni el área 'Jurídica' por sí solas. La prioridad 'Alta' solo es para casos de riesgo legal (fraude, estafa, etc.).
4. Clasifica primero, luego envía el correo con la tool, luego retorna el JSON.
5. El resumen debe ser máximo 100 palabras, objetivo y profesional."""


# --- ChatPromptTemplate ---
# MessagesPlaceholder("agent_scratchpad") es obligatorio para agentes con tools
# Permite que el agente registre su razonamiento intermedio (ReAct pattern)
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_MESSAGE),
    ("human", """
Analiza la siguiente solicitud PQRS:

Nombre: {nombre}
Correo: {correo}
Asunto: {asunto}
Descripción: {descripcion}
Radicado: {radicado}

PASOS OBLIGATORIOS:
1. Analiza el contenido y clasifica la solicitud
2. USA LA HERRAMIENTA enviar_correo_confirmacion para notificar al usuario
3. Retorna el análisis en formato JSON. Asegúrate de que los valores de los campos sean estrictamente los siguientes:
   - categoria: 'Petición', 'Queja', 'Reclamo' o 'Sugerencia'
   - prioridad: 'Alta', 'Media' o 'Baja' (La prioridad 'Alta' es SOLO para casos de riesgo legal directo como fraude, estafa, robo, demanda, lavado de activos. No la uses por simple urgencia o demora.)
   - area: 'Servicio al Cliente', 'Servicios Financieros', 'Operaciones', 'Jurídica' o 'Innovación' (ESTRICTAMENTE UNA DE ESTAS. No uses otros valores ni inventes áreas.)
   - resumen: (máximo 100 palabras)
   - palabras_clave: (lista de palabras críticas *específicamente* definidas como tales: 'fraude', 'estafa', 'robo', 'demanda', 'lavado de activos' si aplica, de lo contrario lista vacía. NO INCLUYAS otras palabras o sinónimos en esta lista.)

   REGLA CRÍTICA Y ABSOLUTA: Si detectas ALGUNA de las palabras clave 'fraude', 'estafa', 'robo', 'demanda' o 'lavado de activos' en el asunto o descripción, DEBES establecer la prioridad a 'Alta' y el área a 'Jurídica'. NO HAY EXCEPCIONES.

Responde ÚNICAMENTE con el JSON, sin texto adicional.
"""),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

print("✅ ChatPromptTemplate configurado")
print("✅ Esquema Pydantic ClasificacionPQRS definido")
print("\nCampos del esquema de salida:")
for field_name, field in ClasificacionPQRS.model_fields.items():
    print(f"   - {field_name}: {field.description}")

✅ ChatPromptTemplate configurado
✅ Esquema Pydantic ClasificacionPQRS definido

Campos del esquema de salida:
   - categoria: Tipo de solicitud: Petición, Queja, Reclamo o Sugerencia
   - prioridad: Nivel de urgencia: Alta, Media o Baja
   - area: Área responsable: Servicio al Cliente, Servicios Financieros, Operaciones, Jurídica o Innovación
   - resumen: Resumen breve del caso en máximo 100 palabras
   - palabras_clave: Lista de las palabras clave *específicamente definidas* como críticas (fraude, estafa, robo, demanda, lavado de activos) si alguna fue detectada. Si no, la lista debe estar vacía.


## 🤖 Celda 8 — Construcción del Agente (AgentExecutor)
Creamos el agente LangChain con `create_tool_calling_agent` + `AgentExecutor`.
Este patrón equivale al nodo **Clasificador PQRS (LangChain Agent)** de n8n.

> **Chain of Thought (ReAct):** El agente usa el patrón Razonar → Actuar → Observar,
> visible en el log de ejecución cuando `verbose=True`.

In [24]:
from langgraph.prebuilt import create_react_agent

# create_react_agent reemplaza al antiguo AgentExecutor + create_tool_calling_agent
# Internamente ejecuta el mismo ciclo ReAct:
# Razonar (LLM) → Actuar (tool) → Observar (resultado) → Razonar de nuevo
agente_executor = create_react_agent(
    model=llm,     # Gemini 2.5 Flash
    tools=tools,   # [enviar_correo_confirmacion]
    # LangGraph maneja el ciclo ReAct automáticamente, sin necesidad de
    # definir max_iterations ni handle_parsing_errors por separado
)

print("✅ Agente ReAct creado con LangGraph")
print("   Tipo de agente: create_react_agent (LangGraph prebuilt)")
print("   LLM: Gemini 2.5 Flash")
print("   Tools disponibles:", [t.name for t in tools])

✅ Agente ReAct creado con LangGraph
   Tipo de agente: create_react_agent (LangGraph prebuilt)
   LLM: Gemini 2.5 Flash
   Tools disponibles: ['enviar_correo_confirmacion']


/tmp/ipykernel_7833/3549500150.py:6: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente_executor = create_react_agent(


## 📊 Celda 9 — Registro en Google Sheets
Función para registrar cada solicitud procesada en la hoja de cálculo.
Equivale al nodo `Registrar en Sheets` de n8n.

In [25]:
def registrar_en_sheets(datos: dict) -> bool:
    """
    Registra una solicitud PQRS procesada en Google Sheets.
    Equivale al nodo 'Registrar en Sheets' de n8n.

    Args:
        datos: Diccionario con todos los campos de la solicitud procesada

    Returns:
        True si se registró exitosamente
    """
    try:
        # Abrimos el spreadsheet por ID
        spreadsheet = gc.open_by_key(SPREADSHEET_ID)

        # Intentamos abrir la hoja
        try:
            hoja = spreadsheet.worksheet(SHEET_NAME)
        except gspread.WorksheetNotFound:
            # If the sheet doesn't exist, create it with predefined headers
            hoja = spreadsheet.add_worksheet(title=SHEET_NAME, rows=1000, cols=10)
            # Use the user's provided headers for the new sheet
            encabezados_usuario = ['radicado', 'nombre', 'correo', 'asunto', 'descripcion', 'fecha', 'categoria', 'prioridad', 'area', 'resumen']
            hoja.append_row(encabezados_usuario)
            print(f"✅ Hoja '{SHEET_NAME}' creada con encabezados definidos por el usuario.")

        # Get actual headers from the sheet to match dynamically
        current_headers = [h.strip().lower() for h in hoja.row_values(1)]

        # Prepare a dictionary to map our data keys to the sheet's column names
        # Standardize keys to match potential sheet headers (lowercase, no spaces, etc.)
        data_to_fill_map = {
            'radicado': datos.get('radicado', ''),
            'fecha': datos.get('fecha', ''),
            'nombre': datos.get('nombre', ''),
            'correo': datos.get('correo', ''),
            'asunto': datos.get('asunto', ''),
            'descripcion': datos.get('descripcion', ''), # This will now come from procesar_pqrs
            'categoria': datos.get('categoria', ''),
            'prioridad': datos.get('prioridad', ''),
            'area': datos.get('area', ''),
            'resumen': datos.get('resumen', ''),
            'palabras clave': ", ".join(datos.get('palabras_clave', [])) # Handle list, match 'palabras clave' header
        }

        # Construct the row (fila) based on the order of current_headers in the sheet
        fila = []
        for header in current_headers:
            # Try to get data using the header itself, or a mapped key
            value = data_to_fill_map.get(header, '')
            fila.append(str(value))

        # --- MODIFICACIÓN: Usar update en lugar de append_row para forzar la escritura en la siguiente fila ---
        # Obtener todas las filas para determinar el número de la última fila con contenido
        all_values = hoja.get_all_values()
        next_row_index = len(all_values) + 1  # +1 porque las filas de gspread son 1-indexed

        print(f"   [DEBUG] Se detectaron {len(all_values)} filas con contenido. Se escribirá en la fila {next_row_index}")

        # Escribir la nueva fila en la siguiente posición disponible
        hoja.update(f'A{next_row_index}', [fila])

        print(f"✅ Solicitud registrada en Google Sheets: {datos.get('radicado')}")
        return True

    except Exception as e:
        print(f"⚠️  Error registrando en Sheets: {e}")
        print("   Continuando sin registro en Sheets...")
        return False


print("✅ Función registrar_en_sheets() lista")

✅ Función registrar_en_sheets() lista


## 🚀 Celda 10 — Pipeline Completo
Aquí unimos todos los componentes en un único flujo, replicando el pipeline completo de n8n:

1. Generar radicado → 2. Agente IA → 3. Verificar prioridad → 4. Alerta jurídica → 5. Registrar en Sheets

In [26]:
import re
import json

def procesar_pqrs(nombre: str, correo: str, asunto: str, descripcion: str) -> dict:
    """
    Pipeline completo de procesamiento PQRS.
    Replica el flujo completo de n8n en un solo función Python.

    Args:
        nombre: Nombre del solicitante
        correo: Correo electrónico del solicitante
        asunto: Asunto de la solicitud
        descripcion: Descripción detallada de la solicitud

    Returns:
        Diccionario con el resultado completo del procesamiento
    """

    print("\n" + "="*60)
    print("🔄 INICIANDO PIPELINE PQRSF")
    print("="*60)

    # ── PASO 1: Generar Radicado ──────────────────────────────────
    # Equivale al nodo 'Generar Radicado' (JavaScript Code en n8n)
    print("\n📌 PASO 1: Generando radicado...")
    info_radicado = generar_radicado()
    radicado = info_radicado["radicado"]
    fecha = info_radicado["fecha"]
    print(f"   Radicado: {radicado}")

    # ── PASO 2: Agente IA (Clasificador PQRS) ────────────────────
    # Equivale al nodo 'Clasificador PQRS' (LangChain Agent en n8n)
    # El agente usa ReAct: razona, llama la tool de Gmail, retorna JSON
    print("\n🤖 PASO 2: Agente LangChain procesando solicitud...")
    print("-" * 40)

    # LangGraph retorna un dict con clave "messages"
    respuesta_agente = agente_executor.invoke({
        "messages": [("human", f"""
    Analiza la siguiente solicitud PQRS:
    Nombre: {nombre}
    Correo: {correo}
    Asunto: {asunto}
    Descripción: {descripcion}
    Radicado: {radicado}

    USA la herramienta enviar_correo_confirmacion. Luego responde SOLO con JSON:
    {{"categoria": "...", "prioridad": "...", "area": "...", "resumen": "...", "palabras_clave": []}}
    """)]
    })

    # El último mensaje es la respuesta final del agente
    agent_output_content = respuesta_agente["messages"][-1].content
    output_raw_json_string = ""

    if isinstance(agent_output_content, list):
        # If content is a list, try to find the JSON string within it
        for item in agent_output_content:
            if isinstance(item, str) and item.strip().startswith('{') and item.strip().endswith('}'):
                output_raw_json_string = item
                break
        if not output_raw_json_string:
            print("⚠️ Agent returned a list content, but no JSON string found. Proceeding with empty string for parsing.")
            # This will lead to JSONDecodeError, which is handled by the except block
    elif isinstance(agent_output_content, str):
        output_raw_json_string = agent_output_content
    else:
        print(f"⚠️ Agent returned unexpected content type: {type(agent_output_content)}. Proceeding with empty string for parsing.")
        # This will lead to JSONDecodeError, which is handled by the except block

    try:
        # Limpiamos posibles bloques de código markdown del LLM
        output_limpio = re.sub(r'```json|```', '', output_raw_json_string).strip()
        clasificacion = json.loads(output_limpio)
    except json.JSONDecodeError:
        # Si falla el parsing, usamos valores por defecto
        print("⚠️  El agente no retornó JSON válido, usando clasificación por defecto")
        clasificacion = {
            "categoria": "Petición",
            "prioridad": "Media",
            "area": "Servicio al Cliente",
            "resumen": descripcion[:100],
            "palabras_clave": []
        }

    categoria = clasificacion.get("categoria", "Petición")
    prioridad = clasificacion.get("prioridad", "Media")
    area = clasificacion.get("area", "Servicio al Cliente")
    resumen = clasificacion.get("resumen", "")
    palabras_clave = clasificacion.get("palabras_clave", [])

    # Flag to remember if critical keywords were detected *before* agent's output is potentially overridden
    critical_keywords_detected_in_original_text = False
    original_text_for_critical_check = (asunto + " " + descripcion).lower()
    predefined_critical_keywords = ["fraude", "estafa", "robo", "demanda", "lavado de activos"]

    if any(keyword in original_text_for_critical_check for keyword in predefined_critical_keywords):
        critical_keywords_detected_in_original_text = True
        print("   🚨 Detectadas palabras clave críticas en el texto original. Forzando área 'Jurídica' y prioridad 'Alta'.")
        area = "Jurídica"
        prioridad = "Alta"
        # Ensure 'palabras_clave' reflects what was found if it wasn't already set by the LLM
        detected_critical_words_from_original = [kw for kw in predefined_critical_keywords if kw in original_text_for_critical_check]
        if not palabras_clave and detected_critical_words_from_original:
            palabras_clave = detected_critical_words_from_original
        elif detected_critical_words_from_original:
            palabras_clave = list(set(palabras_clave + detected_critical_words_from_original))


    # --- NUEVO POST-PROCESAMIENTO ADICIONAL ---

    # 1. Limpiar palabras_clave: Solo permitir las críticas predefinidas
    palabras_clave_filtradas = [kw for kw in palabras_clave if kw in predefined_critical_keywords]
    if len(palabras_clave) != len(palabras_clave_filtradas):
        print(f"   🧹 Limpiando palabras_clave: Eliminadas {list(set(palabras_clave) - set(palabras_clave_filtradas))}")
    palabras_clave = palabras_clave_filtradas


    # 2. Validar área: Asegurarse de que el área sea una de las permitidas
    allowed_areas = ['Servicio al Cliente', 'Servicios Financieros', 'Operaciones', 'Jurídica', 'Innovación']
    if area not in allowed_areas:
        print(f"   ⚠️ Área '{area}' no es válida. Corrigiendo a 'Servicio al Cliente'.")
        area = "Servicio al Cliente"


    # 3. Ajustar prioridad si no hay palabras críticas pero se asignó Alta
    if prioridad == "Alta" and not critical_keywords_detected_in_original_text:
        print("   📉 Prioridad 'Alta' asignada sin palabras clave críticas. Bajando a 'Media'.")
        prioridad = "Media"
    # ----------------------------------------------------------------------------


    print(f"\n✅ Clasificación completada:")
    print(f"   Categoría:  {categoria}")
    print(f"   Prioridad:  {prioridad}")
    print(f"   Área:       {area}")
    if palabras_clave:
        print(f"   ⚠️  Palabras críticas detectadas: {palabras_clave}")

    # ── PASO 3: Verificar Prioridad Alta ─────────────────────────
    # Equivale al nodo 'Verificar Prioridad Alta' (IF node en n8n)
    print(f"\n🔍 PASO 3: Verificando prioridad ({prioridad})...")

    if prioridad == "Alta":
        # ── PASO 3a: Notificar Área Jurídica ──────────────────────
        # Equivale al nodo 'Notificar Área Jurídica' (Gmail node en n8n)
        print("   🚨 Prioridad ALTA detectada — enviando alerta al área jurídica...")

        alerta_html = f"""
        <html><body style=\"font-family: Arial, sans-serif;\">
            <div style=\"background: #e74c3c; padding: 20px; border-radius: 8px;\">
                <h2 style=\"color: white; margin: 0;\">🚨 ALERTA — Caso de Alta Prioridad</h2>
            </div>
            <div style=\"padding: 20px;\">
                <p><strong>Radicado:</strong> {radicado}</p>
                <p><strong>Solicitante:</strong> {nombre} ({correo})</p>
                <p><strong>Categoría:</strong> {categoria}</p>
                <p><strong>Área asignada:</strong> {area}</p>
                <p><strong>Resumen:</strong> {resumen}</p>
                {'<p><strong>⚠️ Palabras críticas:</strong> ' + ', '.join(palabras_clave) + '</p>' if palabras_clave else ''}
                <p><strong>Fecha:</strong> {fecha}</p>
            </div>
        </body></html>
        """

        enviar_correo(
            CORREO_JURIDICA,
            f"🚨 ALERTA URGENTE — {radicado}",
            alerta_html
        )
    else:
        print("   ✅ Prioridad normal — no se requiere alerta urgente")

    # ── PASO 4: Registrar en Google Sheets ───────────────────────
    # Equivale al nodo 'Registrar en Sheets' de n8n
    print("\n📊 PASO 4: Registrando en Google Sheets...")

    datos_registro = {
        "radicado": radicado,
        "fecha": fecha,
        "nombre": nombre,
        "correo": correo,
        "asunto": asunto,
        "descripcion": descripcion,
        "categoria": categoria,
        "prioridad": prioridad,
        "area": area,
        "resumen": resumen,
        "palabras_clave": palabras_clave
    }

    registrar_en_sheets(datos_registro)

    # ── PASO 5: Preparar Resultado Final ─────────────────────────
    # Equivale al nodo 'Preparar Resultado Final' (Set node en n8n)
    resultado = {
        "radicado": radicado,
        "categoria": categoria,
        "prioridad": prioridad,
        "area": area,
        "resumen": resumen,
        "fecha": fecha,
        "nombre": nombre,
        "correo": correo,
        "asunto": asunto
    }

    print("\n" + "="*60)
    print("✅ PIPELINE COMPLETADO")
    print("="*60)

    return resultado


print("✅ Función procesar_pqrs() lista — Pipeline completo configurado")

✅ Función procesar_pqrs() lista — Pipeline completo configurado


## 🧪 Celda 11 — Prueba Caso Normal (Prioridad Media)
Probamos el sistema con una solicitud de ejemplo sin palabras críticas.

In [27]:
# Caso de prueba 1: Queja de servicio al cliente — prioridad esperada: Media
resultado1 = procesar_pqrs(
    nombre="María García López",
    correo="acalebe21@gmail,com",
    asunto="Demora en atención telefónica",
    descripcion="""Llevo más de 30 minutos esperando en línea para hablar con
    un agente de atención al cliente. La línea ha cortado dos veces sin que nadie
    me atienda. Necesito resolver un problema con mi cuenta bancaria con urgencia.
    El servicio ha empeorado notablemente en los últimos meses."""
)

print("\n📋 RESULTADO FINAL:")
for clave, valor in resultado1.items():
    print(f"   {clave}: {valor}")


🔄 INICIANDO PIPELINE PQRSF

📌 PASO 1: Generando radicado...
   Radicado: PQRS-20260608-1780933983378-283

🤖 PASO 2: Agente LangChain procesando solicitud...
----------------------------------------

📨 [MODO DEMO] Correo que se enviaría a: acalebe21@gmail,com
   Asunto: Confirmación de recepción PQRS - PQRS-20260608-1780933983378-283
   Cuerpo (HTML): 
    <html><body style="font-family: Arial, sans-serif; max-width: 600px; margin: 0 auto;">
        <div style="background: linear-gradient(135deg, #667eea, #764ba2); padding: 30px; border-radius: 10p...
   🧹 Limpiando palabras_clave: Eliminadas ['urgencia', 'demora', 'cuenta bancaria', 'servicio', 'atención telefónica']
   ⚠️ Área 'Atención al Cliente' no es válida. Corrigiendo a 'Servicio al Cliente'.
   📉 Prioridad 'Alta' asignada sin palabras clave críticas. Bajando a 'Media'.

✅ Clasificación completada:
   Categoría:  Queja
   Prioridad:  Media
   Área:       Servicio al Cliente

🔍 PASO 3: Verificando prioridad (Media)...
   ✅ Prior

/tmp/ipykernel_7833/1306972297.py:61: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  hoja.update(f'A{next_row_index}', [fila])


✅ Solicitud registrada en Google Sheets: PQRS-20260608-1780933983378-283

✅ PIPELINE COMPLETADO

📋 RESULTADO FINAL:
   radicado: PQRS-20260608-1780933983378-283
   categoria: Queja
   prioridad: Media
   area: Servicio al Cliente
   resumen: El cliente reporta una demora excesiva (más de 30 minutos) en la atención telefónica, con dos cortes de línea, y necesita resolver un problema urgente con su cuenta bancaria, indicando un empeoramiento del servicio.
   fecha: 2026-06-08
   nombre: María García López
   correo: acalebe21@gmail,com
   asunto: Demora en atención telefónica


## 🚨 Celda 12 — Prueba Caso Crítico (Palabras Clave de Riesgo)
Probamos con una solicitud que contiene términos críticos como **"fraude"** o **"estafa"**.
El sistema debe asignar automáticamente prioridad Alta y área Jurídica.

In [28]:
# Caso de prueba 2: Reporte de fraude — prioridad esperada: Alta, área: Jurídica
resultado2 = procesar_pqrs(
    nombre="Carlos Rodríguez Pérez",
    correo="carlos.rodriguez@ejemplo.com",
    asunto="Reporte de fraude en mi cuenta",
    descripcion="""Detecté movimientos no reconocidos en mi cuenta corriente por
    valor de $2.500.000. Creo que fui víctima de una estafa o robo de información
    bancaria. Los cargos se realizaron en ciudades donde no he estado.
    Exijo investigación inmediata y me reservo el derecho de interponer una demanda
    si no hay respuesta en 24 horas."""
)

print("\n📋 RESULTADO FINAL:")
for clave, valor in resultado2.items():
    print(f"   {clave}: {valor}")

# Verificación automática
print("\n🔍 Verificación:")
assert resultado2['prioridad'] == 'Alta', "❌ La prioridad debería ser Alta"
assert resultado2['area'] == 'Jurídica', "❌ El área debería ser Jurídica"
print("   ✅ Prioridad Alta asignada correctamente")
print("   ✅ Área Jurídica asignada correctamente")


🔄 INICIANDO PIPELINE PQRSF

📌 PASO 1: Generando radicado...
   Radicado: PQRS-20260608-1780934086595-060

🤖 PASO 2: Agente LangChain procesando solicitud...
----------------------------------------

📨 [MODO DEMO] Correo que se enviaría a: carlos.rodriguez@ejemplo.com
   Asunto: Confirmación de recepción PQRS - PQRS-20260608-1780934086595-060
   Cuerpo (HTML): 
    <html><body style="font-family: Arial, sans-serif; max-width: 600px; margin: 0 auto;">
        <div style="background: linear-gradient(135deg, #667eea, #764ba2); padding: 30px; border-radius: 10p...
   🚨 Detectadas palabras clave críticas en el texto original. Forzando área 'Jurídica' y prioridad 'Alta'.
   🧹 Limpiando palabras_clave: Eliminadas ['investigación', 'cuenta corriente', 'movimientos no reconocidos', 'robo de información']

✅ Clasificación completada:
   Categoría:  Reclamo
   Prioridad:  Alta
   Área:       Jurídica
   ⚠️  Palabras críticas detectadas: ['demanda', 'estafa', 'fraude', 'robo']

🔍 PASO 3: Verifican

/tmp/ipykernel_7833/1306972297.py:61: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  hoja.update(f'A{next_row_index}', [fila])


## 📈 Celda 13 — Resumen de Casos Procesados
Visualizamos un resumen de todos los casos procesados en esta sesión.

In [29]:
# Recopilamos los resultados de las pruebas restantes
resultados = [resultado1, resultado2]

print("\n" + "="*80)
print("📊 RESUMEN DE CASOS PROCESADOS")
print("="*80)
print(f"{'Radicado':<40} {'Categoría':<12} {'Prioridad':<10} {'Área':<25}")
print("-"*80)

for r in resultados:
    # Mostramos el radicado abreviado para que quepa en la tabla
    radicado_corto = r['radicado'][:38] + ".." if len(r['radicado']) > 38 else r['radicado']
    print(f"{radicado_corto:<40} {r['categoria']:<12} {r['prioridad']:<10} {r['area']:<25}")

print("="*80)
print(f"Total casos procesados: {len(resultados)}")
print(f"Casos Alta prioridad:   {sum(1 for r in resultados if r['prioridad'] == 'Alta')}")
print(f"Casos Media prioridad:  {sum(1 for r in resultados if r['prioridad'] == 'Media')}")


📊 RESUMEN DE CASOS PROCESADOS
Radicado                                 Categoría    Prioridad  Área                     
--------------------------------------------------------------------------------
PQRS-20260608-1780933983378-283          Queja        Media      Servicio al Cliente      
PQRS-20260608-1780934086595-060          Reclamo      Alta       Jurídica                 
Total casos procesados: 2
Casos Alta prioridad:   1
Casos Media prioridad:  1


---

## 📐 Resumen de la Arquitectura LangChain (para la diapositiva 07)

| Componente | Implementación | Equivalente en n8n |
|---|---|---|
| **LLM** | `ChatGoogleGenerativeAI` — Gemini 2.5 Flash, temperature=0 | OpenAI Chat Model (GPT-4o-mini) |
| **ChatPromptTemplate** | System message + Human template + `MessagesPlaceholder` | Prompt del nodo Clasificador PQRS |
| **Tools** | `@tool enviar_correo_confirmacion` | Gmail Tool |
| **Agent** | `create_tool_calling_agent` (ReAct pattern) | LangChain Agent |
| **AgentExecutor** | Ciclo Razonar → Actuar → Observar, max 5 iteraciones | Execution engine de n8n |
| **Output Parser** | `json.loads()` + Pydantic `ClasificacionPQRS` | Structured Output Parser |
| **Persistencia** | `gspread` → Google Sheets | Nodo Google Sheets |

## 🧠 Cadena de Pensamiento (para la diapositiva 08)

El agente usa el patrón **ReAct (Reason + Act)**:
1. **Razona** sobre el contenido de la solicitud
2. **Actúa** llamando la tool `enviar_correo_confirmacion`
3. **Observa** el resultado de la tool (correo enviado)
4. **Razona** de nuevo y retorna el JSON estructurado

El log completo de este ciclo es visible gracias a `verbose=True` en el `AgentExecutor`.

**No se implementa RAG** porque la clasificación PQRS se basa en reglas explícitas en el system message, no en recuperación de documentos externos. El conocimiento del dominio está embebido en el prompt.